In [ ]:
#  Git link: https://github.com/dayanaviana/WGU/tree/main/D212-DataMiningII

In [ ]:
from platform import python_version
("Pyhton version:", python_version())

## Imports

In [ ]:
import numpy as np 
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

In [ ]:
filePath = "../datasources/churn.csv"
df_raw = pd.read_csv(filePath, index_col=False)
df_raw.shape #(Rows, Columns)

In [ ]:
### Select numeric features
numeric_features = df_raw.select_dtypes(include=['int64','float64']).columns.to_list()
print("count:", len(numeric_features))
print("numeric_features list ", numeric_features)

In [ ]:
### Remove features
df_reduced = df_raw[numeric_features]
df_reduced = df_reduced.drop(columns=['CaseOrder','Zip', 'Lat', 'Lng',
                                      'Item1', 'Item2', 'Item3', 'Item4', 'Item5', 'Item6', 'Item7', 'Item8'])
df_reduced.head()

## Data Exploration

In [ ]:
df_reduced.describe().round(2)

In [ ]:
plt.figure(figsize=(6, 3))

df_reduced.boxplot()

plt.xticks(
    rotation=30,
    ha='right',   # alinhamento
    fontsize=9
)

plt.tight_layout()
plt.show()

# Visualize Correlations

In [ ]:
sns.pairplot(df_reduced, diag_kind='hist')

In [ ]:
cmap = sns.diverging_palette(h_neg=10,
h_pos=240,
as_cmap=True)

corr = df_reduced.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(df_reduced.corr(), mask=mask,
center=0, cmap=cmap, linewidths=1,
annot=True, fmt=".2f")

In [ ]:
# Both features bring the same information, as one increase the other one also increases
ax = sns.scatterplot(data = df_reduced,
                     x = 'Tenure',
                     y = 'Bandwidth_GB_Year',
                     s = 50
                     )

In [ ]:
# Remove the redundant feature
df_reduced = df_reduced.drop('Bandwidth_GB_Year', axis=1)

## Data Standardization

In [ ]:
# SCALING: Normalize data using z-score from sklearn StandardScaler

# https://www.w3schools.com/python/python_ml_scale.asp
 
from sklearn.preprocessing import StandardScaler 
scaler = StandardScaler() 

# Scale numeric data
a_ndarray = scaler.fit_transform(df_reduced) 

# Transform array into a Data Frame
df_standardized = pd.DataFrame(a_ndarray, columns=df_reduced.columns)

# Clear variables
del scaler, a_ndarray, 

df_standardized.describe().round(2)

In [ ]:
df_standardized.info()

In [ ]:
# Write normalized data to file
df_standardized.to_csv("df_normalized_D212_2_PCA.csv")

Link to a copy of the cleaned dataset: 
https://raw.githubusercontent.com/dayanaviana/WGU/refs/heads/main/D212-DataMiningII/df_normalized_D212_2_PCA.csv 


In [ ]:
df_standardized.boxplot()

plt.xticks(
    rotation=30,
    ha='right',   # alinhamento
    fontsize=9
)

plt.tight_layout()
plt.show()

## T-SNE
- t‑SNE does not preserve global structure
- Distances between clusters are not meaningful
- Results vary with random seed and hyperparameters
- It is a visualization tool, not a clustering algorithm

In [ ]:
# Visually explore the patterns in a high dimensional dataset.

from sklearn.manifold import TSNE
m = TSNE(learning_rate=50)

tsne_features = m.fit_transform(df_standardized)
tsne_features[1:4,:]

df_raw['x'] = tsne_features[:,0]
df_raw['y'] = tsne_features[:,1]

sns.scatterplot(x="x", y="y", hue='Churn', data=df_raw)
plt.show()

This t‑SNE visualization shows partial separation between churn and non‑churn customers, 

but with substantial overlap, indicating that churn behavior is complex and influenced by multiple interacting variables.

Dimensionality reduction analysis is required to fully understand customer bahavior here.

# The Model

In [ ]:
pca = PCA()

# Fit PCA
pca.fit(df_standardized)

# Matrix of all principal components (loadings)
model_matrix = pca.components_.T
model_matrix

## Loading Matrix

In [ ]:
# pca.components_ gives components as rows, the transpose gives the standard format:

pca_matrix_df = pd.DataFrame(
    pca.components_.T,
    index=df_standardized.columns,
    columns=[f"PC{i+1}" for i in range(len(df_standardized.columns))]
)

pca_matrix_df

# Check optimal number of principal components

In [ ]:
explained_variance = pca.explained_variance_ratio_
explained_variance

In [ ]:
# Elbow method
plt.figure(figsize=(8,5))
plt.plot(range(1, len(explained_variance)+1), 
         explained_variance, 
         marker='o')
plt.title("Scree Plot")
plt.xlabel("Principal Component")
plt.ylabel("Variance Explained")
plt.grid(True)
plt.show()

In [ ]:
eigenvalues = pca.explained_variance_
eigenvalues

In [ ]:
# Keise Criterion
sum(eigenvalues > 1)

In [ ]:
# create an array that starts with 1, instead of 0
pcomp = np.arange(pca.n_components_)
pcomp

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(pcomp,
         explained_variance,
         'b')
plt.title("Scree Plot (Kaiser Criterion)")
plt.xlabel("Number of Component")
plt.ylabel("Variance Explained / Eigenvalues")
plt.grid(True)
plt.show()

### Total variance

In [ ]:
# variance of the first four principal components
each_variance = pca.explained_variance_[:4]
each_variance

In [ ]:
total_variance_captured = np.sum(pca.explained_variance_ratio_[:4])
total_variance_captured

In [ ]:
total_variance = sum(pca.explained_variance_ratio_) * 100
total_variance

## Analyze and interpret results

In [ ]:
pc4 = pca_matrix_df[["PC1","PC2","PC3","PC4"]]
pc4

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

# Create a 2‑color diverging blue colormap
two_blue_cmap = LinearSegmentedColormap.from_list(
    "two_blue",
    ["#08306B", "#DEEBF7", "#08306B"]  # dark blue → light blue → dark blue
)

plt.figure(figsize=(10,8))
sns.heatmap(
    pca_matrix_df.iloc[:, :4],
    annot=True,
    cmap=two_blue_cmap,
    center=0
)
plt.title("PCA Loading Matrix (PC1–PC4)")
plt.show()
